# Single-spectrum BH fit debug (shot 193788, frame 8, channel 5)

This notebook validates the **shared fitting preprocessing** that batch and single-spectrum debug now agree on:

1. Load the raw row from the FITS cube for `(FRAME, CHANNEL)`.
2. Subtract the mean over `BACKGROUND_FRAMES` for the same channel.
3. Crop to the BH **fit** wavelength window `BH_FIT_WAVELENGTH_RANGE_NM`.
4. Compute a **positive** normalization scale strictly inside `BH_SCALE_WAVELENGTH_RANGE_NM` (so unrelated bright features such as H-γ cannot dominate).
5. Divide by that scale.  **Negative values are preserved.**
6. Fit with the BH model, with the constant baseline parameter `base` **tightly bounded near zero**.

Display-grid normalization (`normalize_curves_for_grid`) is a separate step and is **not** used by the fitter.

In [ ]:
from pathlib import Path

FITS_FILE = Path("~/Dropbox/Experiments/2025-LHD-BH/133mORCA/193788.fits").expanduser()
SHOT = 193788
FRAME = 8
CHANNEL = 5
BACKGROUND_FRAMES = (0, 1, 2, 3)

BH_FIT_WAVELENGTH_RANGE_NM = (433.05, 433.90)
BH_SCALE_WAVELENGTH_RANGE_NM = (433.08, 433.30)

CW_NM = 431.91
SCALE = 1.0
TIME_RANGE = (0.0, 10.0)
DARK_FRAME = None

SAVE_DEBUG_FIGURES = False
DEBUG_OUT = Path("debug_single_spectrum")

if SAVE_DEBUG_FIGURES:
    DEBUG_OUT.mkdir(parents=True, exist_ok=True)

if not FITS_FILE.is_file():
    raise FileNotFoundError(
        f"FITS not found: {FITS_FILE}\n"
        "Adjust FITS_FILE or run this notebook on a host where the data exist."
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from bh_molecule.fit import BHFitter
from bh_molecule.physics import BHModel
from bh_molecule.workflows.batch_fit import prepare_vis_for_bh_batch
from bh_molecule.workflows.preprocessing import (
    mean_background_spectrum,
    make_bh_fit_preprocessor,
    prepare_bh_fit_arrays,
)

vis = prepare_vis_for_bh_batch(
    FITS_FILE,
    cw=CW_NM,
    scale=SCALE,
    dark_frame=DARK_FRAME,
    time_range=TIME_RANGE,
)
F, C, P = vis.cube.shape
print(f"Loaded vis cube shape: F={F} C={C} P={P}")
print(f"Requested BACKGROUND_FRAMES: {BACKGROUND_FRAMES}")
print(f"BH fit window:   {BH_FIT_WAVELENGTH_RANGE_NM} nm")
print(f"BH scale window: {BH_SCALE_WAVELENGTH_RANGE_NM} nm")

In [ ]:
x_fit, y_fit, meta = prepare_bh_fit_arrays(
    vis,
    FRAME,
    CHANNEL,
    background_frames=BACKGROUND_FRAMES,
    fit_window=BH_FIT_WAVELENGTH_RANGE_NM,
    scale_window=BH_SCALE_WAVELENGTH_RANGE_NM,
)

print("--- preprocessing metadata ---")
for k, v in meta.items():
    print(f"  {k}: {v}")
print()
print(f"Actual background frames used: {meta['background_frames']}")
print(f"Scale window used:             {meta['scale_window']} nm")
print(f"Scale value used:              {meta['scale']:.6g}")
print(f"Raw min/max/median (fit window):        "
      f"{meta['raw_min']:.4g} / {meta['raw_max']:.4g} / {meta['raw_median']:.4g}")
print(f"Background min/max/median (fit window): "
      f"{meta['background_min']:.4g} / {meta['background_max']:.4g} / {meta['background_median']:.4g}")
print(f"Subtracted min/max/median (fit window): "
      f"{meta['subtracted_min']:+.4g} / {meta['subtracted_max']:+.4g} / {meta['subtracted_median']:+.4g}")
print(f"Normalized min/max/median (fit window): "
      f"{meta['normalized_min']:+.4g} / {meta['normalized_max']:+.4g} / {meta['normalized_median']:+.4g}")
print(f"Negative points after subtraction:      {meta['n_negative_after_subtract']}")

In [ ]:
preprocess = make_bh_fit_preprocessor(
    background_frames=BACKGROUND_FRAMES,
    fit_window=BH_FIT_WAVELENGTH_RANGE_NM,
    scale_window=BH_SCALE_WAVELENGTH_RANGE_NM,
)

model = BHModel()
fitr = BHFitter(
    vis=vis,
    model=model,
    preprocess=preprocess,
    base_tight=True,
    nm_window=BH_FIT_WAVELENGTH_RANGE_NM,
)

base_idx = fitr.param_names.index("base")
print("--- baseline / offset bounds (after preprocessing) ---")
print(f"  parameter: {fitr.param_names[base_idx]}")
print(f"  p0:        {fitr.p0[base_idx]:+.4f}")
print(
    f"  bounds:    [{fitr.bounds[0][base_idx]:+.4f}, "
    f"{fitr.bounds[1][base_idx]:+.4f}]"
)

try:
    res = fitr.fit(FRAME, CHANNEL, return_fit=True)
    fit_ok = True
    fit_err = None
except Exception as exc:
    res = None
    fit_ok = False
    fit_err = repr(exc)

if fit_ok:
    print("\n--- fit result ---")
    dx_idx = fitr.param_names.index("dx")
    print(f"  fit success: True")
    print(f"  fitted dx = {res['params'][dx_idx]:+.5f} nm")
    for n, v in zip(fitr.param_names, res["params"]):
        print(f"    {n} = {v:+.6g}")
    resid = res["y"] - res["yfit"]
    rms = float(np.sqrt(np.mean(resid ** 2)))
    print(f"  residual rms = {rms:.5f}")
    print(f"  residual max|.| = {float(np.max(np.abs(resid))):.5f}")
    print(f"  yfit max = {float(res['yfit'].max()):.4f}  (should be ~1)")
else:
    print(f"\nFit FAILED: {fit_err}")

In [ ]:
wl_chan = np.asarray(vis.wl_nm[CHANNEL], dtype=float)
row_raw = np.asarray(vis.cube[FRAME, CHANNEL], dtype=float)
bg_row = mean_background_spectrum(vis.cube, BACKGROUND_FRAMES, CHANNEL)
y_sub_full = row_raw - bg_row

nm_lo, nm_hi = BH_FIT_WAVELENGTH_RANGE_NM
sc_lo, sc_hi = BH_SCALE_WAVELENGTH_RANGE_NM
fit_mask = (wl_chan >= nm_lo) & (wl_chan <= nm_hi)

fig, axes = plt.subplots(5, 1, figsize=(9, 12), sharex=False)

ax = axes[0]
ax.plot(wl_chan, row_raw, "k-", lw=0.8, label="raw full channel spectrum")
ax.axvspan(nm_lo, nm_hi, color="#7397de", alpha=0.10, label="BH fit window")
ax.axvspan(sc_lo, sc_hi, color="orange", alpha=0.20, label="BH scale window")
ax.set_ylabel("raw counts")
ax.set_xlabel("wavelength [nm]")
ax.set_title(f"1) raw full spectrum  (shot {SHOT} f{FRAME} ch{CHANNEL})")
ax.legend(fontsize=8, loc="best")

ax = axes[1]
ax.plot(wl_chan[fit_mask], row_raw[fit_mask], "k.", ms=2, label="raw (fit window)")
ax.axvspan(sc_lo, sc_hi, color="orange", alpha=0.20, label="BH scale window")
ax.set_xlim(nm_lo, nm_hi)
ax.set_ylabel("raw counts")
ax.set_title("2) raw BH-window spectrum")
ax.legend(fontsize=8)

ax = axes[2]
ax.plot(wl_chan[fit_mask], bg_row[fit_mask], "r-", lw=1, label=f"mean(BG frames={BACKGROUND_FRAMES})")
ax.set_xlim(nm_lo, nm_hi)
ax.set_ylabel("raw counts")
ax.set_title("3) background BH-window spectrum")
ax.legend(fontsize=8)

ax = axes[3]
ax.plot(wl_chan[fit_mask], y_sub_full[fit_mask], "g.", ms=2, label="raw - background")
ax.axhline(0, color="gray", lw=0.7, ls=":")
ax.axvspan(sc_lo, sc_hi, color="orange", alpha=0.20, label="BH scale window")
ax.set_xlim(nm_lo, nm_hi)
ax.set_ylabel("counts")
ax.set_title("4) background-subtracted BH-window spectrum (negatives preserved)")
ax.legend(fontsize=8)

ax = axes[4]
ax.plot(x_fit, y_fit, "k.", ms=3, label="data (preprocessed)")
if fit_ok:
    ax.plot(res["x"], res["yfit"], "-", color="#7397de", lw=1.5, label="BH fit")
ax.axhline(0, color="gray", lw=0.7, ls=":")
ax.axvspan(sc_lo, sc_hi, color="orange", alpha=0.20, label="BH scale window")
ax.set_xlim(nm_lo, nm_hi)
ax.set_xlabel("wavelength [nm]")
ax.set_ylabel("normalized intensity")
ax.set_title("5) normalized-for-fit BH-window spectrum + final fit")
ax.legend(fontsize=8)

fig.tight_layout()
if SAVE_DEBUG_FIGURES:
    fig.savefig(DEBUG_OUT / f"{SHOT}_f{FRAME}_ch{CHANNEL}_debug.png", dpi=150)
plt.show()

## Summary

- `BACKGROUND_FRAMES = (0, 1, 2, 3)` is averaged from `vis.cube` and subtracted from the row.
- The scale used by the fitter is the maximum positive value **inside `BH_SCALE_WAVELENGTH_RANGE_NM` only**, so H-γ (or any bright line outside that sub-window) cannot dominate the BH scale.
- The constant baseline `base` is bounded to `±0.03` after preprocessing.
- The final fit on shot 193788 frame 8 channel 5 reproduces the BH band without any manual `set_scale(0.0001837)` compensation.